# Grevy's zebra body orientation during an escape response

A focal group of Grevy's zebra (*Equus grevyi*, $N = 18$) was recorded with an overhead drone at the Mpala Conservancy in Laikipia, Kenya: the body orientations come from the drone recordings of B. Koger, A. Deshpande, J. T. Kerby, J. M. Graving, B. R. Costelloe, and I. D. Couzin, "Quantifying the movement, behaviour and environmental context of group-living animals using drones and computer vision," *Journal of Animal Ecology* **92**(6), 1415-1433 (2023); [doi:10.1111/1365-2656.13904](https://doi.org/10.1111/1365-2656.13904). The first two flights captured baseline (grazing) behavior; the third flight contains a disturbance event in which a human observer approaches the group. Body orientation is available for each frame from automated tracking (body axis from the tail base to the neck base; tracked with DeepPoseKit, J. M. Graving et al., *eLife* **8**, e47994 (2019); [doi:10.7554/eLife.47994](https://doi.org/10.7554/eLife.47994)).

To reduce temporal autocorrelation, orientations are circularly averaged over
non-overlapping 300-frame windows (10 s at 30 Hz), giving approximately
independent observations. Each window is labeled
**pre-approach** (the pre-stimulus baseline) or
**post-approach** (the post-stimulus escape response), so the data
are repeated direction measurements nested within individuals.

**Scientific question.** Does the group's typical orientation change in response
to the stimulus, and how precisely do individuals orient? The hierarchical model
separates three complementary aspects of the response:

- a group-level shift in the mean direction ($\mu$);
- **between-individual alignment**: how tightly the individual mean directions
  cluster, summarized as a mean resultant length in $[0, 1]$;
- **within-individual concentration**: $\kappa$ per track, reported on a
  $[0, 1]$ scale via the Bessel ratio $R(\kappa) = I_1(\kappa)/I_0(\kappa)$.


This R port of `notebooks/zebra_escape.ipynb` fits the same joint
hierarchical von Mises model with `brms` and the custom
`unwrap_von_mises` family. Both mean direction and concentration have
`PostStimulus` effects and individual `Track` intercept/slope effects.


## Setup

Follow the [R setup](../getting-started-r.md) page once, select the
**R (`ir`)** kernel, and run all cells in order. The setup cell restores
missing packages and CmdStan. This draft is intentionally unbaked;
executing it records versions, diagnostics, inline PNGs, and vector PDFs.


In [ ]:
# Setup: make sure this notebook's stack is installed. The Python
# notebooks rely on the local virtual environment (or Colab's bootstrap
# cell); R has no one-click runtime, so this cell guarantees the packages
# instead. Follow the R setup page for the manual route.
# cmdstanr is distributed by the Stan team on r-universe (it is no
# longer on CRAN); everything else comes from CRAN.
if (!requireNamespace("cmdstanr", quietly = TRUE)) {
  # two repos so cmdstanr's CRAN dependencies resolve as well
  install.packages("cmdstanr",
    repos = c("https://stan-dev.r-universe.dev", "https://cloud.r-project.org"))
}
required <- c("brms", "posterior", "loo", "circular", "ggplot2", "ragg", "arrow")
missing <- setdiff(required, rownames(installed.packages()))
if (length(missing)) {
  install.packages(missing, repos = "https://cloud.r-project.org")
}
cmdstan_ok <- tryCatch(dir.exists(as.character(cmdstanr::cmdstan_path())),
  error = function(e) FALSE)
if (!cmdstan_ok) {
  # One-time ~50 MB download (see the R setup page).
  cmdstanr::install_cmdstan()
}


In [ ]:
suppressPackageStartupMessages({
    library(brms)
    library(posterior)
    library(loo)
    library(circular)
    library(ggplot2)
    library(arrow)
})

# The repository's R helpers (custom brms family + circular statistics)
# live next to the notebook; make them reachable regardless of the
# kernel's working directory.
repo_root <- if (file.exists(file.path("..", "R", "unwrap_family.R"))) ".." else
    if (file.exists(file.path("R", "unwrap_family.R"))) "." else
    stop("Run this notebook from notebooks_r/ or from the repository root.")
source(file.path(repo_root, "R", "unwrap_family.R"))
source(file.path(repo_root, "R", "unwrap_functions.R"))

# The figure code writes to the repository-relative figures directory.
FIG_DIR <- file.path(repo_root, "figures", "zebra_escape")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)
theme_set(theme_minimal(base_size = 12))

# Headless R builds (like this CI container) ship with neither X11 nor
# cairo, so the kernel's default device cannot rasterise plots. Render
# each figure to PNG with ragg (self-contained; no system graphics
# stack required) and display it inline; the vector PDFs remain the
# archival copies.
show_figure <- function(plot, name, width, height, res = 110) {
  file <- file.path(FIG_DIR, paste0(name, ".png"))
  ragg::agg_png(file, width = width, height = height, units = "in",
                res = res)
  print(plot)
  invisible(dev.off())
  IRdisplay::display_png(file = file)
  invisible(NULL)
}


SEED <- 42
set.seed(SEED)
ci_run <- Sys.getenv("UNWRAP_R_CI") == "1"
CHAINS <- if (ci_run) 2L else 4L
# Hierarchical models need longer adaptation and more retained draws than
# the introductory GLM. CI keeps the full data and model, shortening only
# sampling and Monte Carlo plotting/bootstrapping budgets.
ITER <- if (ci_run) 3000L else 3000L
WARMUP <- if (ci_run) 1500L else 2000L
ADAPT_DELTA <- 0.97
# The tighter adapt_delta protects the grouped concentration geometry;
# these are execution budgets, not a claim of verified convergence.

c(R = as.character(getRversion()),
  brms = as.character(packageVersion("brms")),
  cmdstanr = as.character(packageVersion("cmdstanr")),
  posterior = as.character(packageVersion("posterior")),
  loo = as.character(packageVersion("loo")),
  circular = as.character(packageVersion("circular")),
  ggplot2 = as.character(packageVersion("ggplot2")),
  ragg = as.character(packageVersion("ragg")),
  arrow = as.character(packageVersion("arrow")))


## Figure style

Figures render inline with the tutorial's `show_figure()` helper and
save vector PDFs under `figures/zebra_escape/`. The condition colors match
the Python analysis. Font selection uses the device's portable sans
family. All angular calculations use radians and circular summaries.


In [ ]:
circ_mean <- function(a) atan2(mean(sin(a)), mean(cos(a)))
resultant <- function(a) Mod(mean(exp(1i * a)))
circ_sd <- function(a, m = circ_mean(a)) {
  sqrt(-2 * log(min(1, Mod(mean(exp(1i * (a - m)))))))
}
# Scaled Bessel functions avoid overflow when concentrations are large.
A1_safe <- function(k) {
  out <- besselI(k, 1, expon.scaled = TRUE) /
    besselI(k, 0, expon.scaled = TRUE)
  if (is.matrix(k)) dim(out) <- dim(k)
  out
}
vm_logpdf <- function(y, mu, kappa) {
  kappa * (cos(y - mu) - 1) - log(2 * pi) -
    log(besselI(kappa, 0, expon.scaled = TRUE))
}
hdi <- function(x, prob = 0.95) {
  x <- sort(as.numeric(x))
  stopifnot(length(x) > 1L, all(is.finite(x)))
  gap <- min(length(x) - 1L, floor(prob * length(x)))
  starts <- seq_len(length(x) - gap)
  j <- starts[which.min(x[starts + gap] - x[starts])]
  c(low = x[j], high = x[j + gap])
}
interval_row <- function(x, angular = FALSE, prob = 0.95) {
  m <- if (angular) circ_mean(x) else mean(x)
  # Unwrap angular intervals around their circular mean before drawing;
  # display endpoints can lie just beyond +/-pi to avoid a false long arc.
  z <- if (angular) m + mod_circular(x - m) else x
  c(mean = m, hdi(z, prob))
}
inverse_A1 <- function(r) {
  if (r <= 0) {
    return(0)
  }
  if (r >= 1) {
    return(Inf)
  }
  upper <- max(2, 1 / (1 - r))
  uniroot(function(k) A1_safe(k) - r, c(0, upper), tol = 1e-9)$root
}
# Exact tutorial sampler convention: scalar mu/kappa, then wrap the draw.
vmu <- function(m) circular::circular(
    m, units = "radians", template = "none", modulo = "2pi")
vm_ctrl <- list(units = "radians", template = "none", modulo = "2pi")
simulate_angles <- function(mu, kappa) {
  stopifnot(is.matrix(mu), identical(dim(mu), dim(kappa)))
  replicated <- matrix(0, nrow = nrow(mu), ncol = ncol(mu))
  for (d in seq_len(nrow(mu))) {
    for (i in seq_len(ncol(mu))) {
      replicated[d, i] <- as.numeric(circular::rvonmises(1,
        mu = vmu(mu[d, i]), kappa = kappa[d, i],
        control.circular = vm_ctrl))
    }
  }
  mod_circular(replicated)
}
save_figure <- function(plot, name, width = 8, height = 4) {
  ggsave(file.path(FIG_DIR, paste0(name, ".pdf")), plot,
         width = width, height = height)
  show_figure(plot, name, width, height)
}

# Same published source files as the Python loaders; keep their local
# checkout under the repository root, reachable from notebooks_r/ too.
# An existing checkout can be supplied through UNWRAP_DATA_ROOT.
data_root <- Sys.getenv("UNWRAP_DATA_ROOT",
                        unset = file.path(repo_root, "unwrap-data"))
ensure_data <- function(subdir) {
  target <- file.path(data_root, subdir)
  if (!dir.exists(target)) {
    if (!dir.exists(data_root)) {
      status <- system2("git", c("clone", "--depth", "1",
        "--filter=blob:none", "--sparse",
        "https://github.com/jgraving/unwrap-data.git", shQuote(data_root)))
      if (status != 0L) stop("Could not fetch the published study data.")
    }
    # Sparse checkout downloads only this study; later notebooks may add
    # their own subdirectory to the same checkout without replacing it.
    status <- system2("git", c("-C", shQuote(data_root),
                              "sparse-checkout", "add", subdir))
    if (status != 0L) stop("Could not materialize the study subdirectory.")
  }
  stopifnot(dir.exists(target))
  target
}


## 1. Load the block-averaged bearings

Read the same published `zebra_bearings_blocks.parquet` as the Python
loader. Each row is a non-overlapping 300-frame (10 s at 30 Hz)
circular average, $\operatorname{atan2}(\overline{\sin b},\overline{\cos b})$.
The published table already excludes tracks absent from either condition;
do not average the blocks again. `Track` labels the 18 individuals and
`PostStimulus` codes pre-approach (0) versus post-approach (1).


In [ ]:
data_dir <- ensure_data("zebra_bearings")
data <- as.data.frame(arrow::read_parquet(file.path(data_dir,
                                              "zebra_bearings_blocks.parquet")))
needed <- c("Track", "Observation", "Condition", "HeadingRadians", "PostStimulus")
stopifnot(all(needed %in% names(data)), !anyNA(data[needed]))
data$Track <- factor(data$Track)
data$PostStimulus <- as.integer(data$PostStimulus)
data$HeadingRadians <- mod_circular(data$HeadingRadians)
condition_labels <- c("pre-approach", "post-approach")
condition_colors <- c("pre-approach" = "#E69F00", "post-approach" = "#7B3294")
data$condition <- factor(data$PostStimulus, levels = 0:1, labels = condition_labels)
stopifnot(nlevels(data$Track) == 18L,
          all(table(data$Track, data$PostStimulus) > 0),
          all(is.finite(data$HeadingRadians)))
head(data)


## 2. Specify the joint model

Both submodels match `HeadingRadians ~ PostStimulus +
(1 + PostStimulus | Track)` and `kappa ~ PostStimulus +
(1 + PostStimulus | Track)` in the Python notebook. The nonlinear
direction formula preserves the centered circular effect priors from
the Rmd's grouped-data machinery.

Mean-direction priors: intercept `VonMises(0, 1e-6)`, stimulus
`VonMises(0, 2)`; independent individual intercept/slope deviations
`VonMises(0, kappa)` with separate `LogNormal(log(30), 0.5)` hyperpriors.
Concentration priors (pre-softplus): intercept
`Normal(inverse_softplus(2), 1)`, stimulus `Normal(0, 2)`, independent
Normal individual deviations with `HalfNormal(2)` scales. Explicit
concentration intercept coding prevents predictor centering, matching
`center_predictors=False` on the Python side.


In [ ]:
# Preserve independent circular intercept and slope deviations. brms's
# ordinary (1 + PostStimulus | Track) mu term would impose Gaussian effects.
# Following the Rmd, represent them as nonlinear coefficient vectors.
form_vonmises <- bf(HeadingRadians ~ mu,
  nlf(mu ~ beta0 + bias0 + PostStimulus * bias1),
  beta0 ~ 1 + PostStimulus,
  bias0 ~ 0 + Track,
  bias1 ~ 0 + Track,
  kappa ~ 0 + Intercept + PostStimulus + (1 + PostStimulus || Track),
  family = unwrap_von_mises)

# One LogNormal(log(30), 0.5) hyperprior for each direction-effect vector,
# as in Python. Direct positive parameters need no softplus Jacobian.
stan_direction_priors <- stanvar(scode = "
  real<lower=0> kmu_intercept;
  real<lower=0> kmu_slope;
", block = "parameters") + stanvar(scode = "
  real unwrap_von_mises_vect_lpdf(vector y, real mu, real kappa) {
    real lp = 0;
    for (i in 1:num_elements(y)) {
      lp += unwrap_von_mises_lpdf(y[i] | mu, kappa);
    }
    return lp;
  }
", block = "functions")
prior_vonmises <- (
  # Bounds identify one period and match PyMC's VonMises support.
  # Uniform factors add constants only. Apply the von Mises log densities
  # explicitly so brms does not request nonexistent custom-prior CDFs
  # when normalizing a bounded coefficient prior.
  prior("uniform(-pi(), pi())", class = "b", nlpar = "beta0", lb = -pi, ub = pi) +
  prior("uniform(-pi(), pi())", class = "b", nlpar = "bias0", lb = -pi, ub = pi) +
  prior("uniform(-pi(), pi())", class = "b", nlpar = "bias1", lb = -pi, ub = pi) +
  set_prior("target += unwrap_von_mises_lpdf(b_beta0[1] | 0, 1e-6)", check = FALSE) +
  set_prior("target += unwrap_von_mises_lpdf(b_beta0[2] | 0, 2)", check = FALSE) +
  set_prior("target += unwrap_von_mises_vect_lpdf(b_bias0 | 0, kmu_intercept)",
            check = FALSE) +
  set_prior("target += unwrap_von_mises_vect_lpdf(b_bias1 | 0, kmu_slope)",
            check = FALSE) +
  set_prior("target += lognormal_lpdf(kmu_intercept | log(30), 0.5)",
            check = FALSE) +
  set_prior("target += lognormal_lpdf(kmu_slope | log(30), 0.5)",
            check = FALSE) +
  prior("normal(log(exp(2) - 1), 1)", class = "b", dpar = "kappa",
        coef = "Intercept") +
  prior("normal(0, 2)", class = "b", dpar = "kappa",
        coef = "PostStimulus") +
  prior("normal(0, 2)", class = "sd", dpar = "kappa")
)
# EQUIVALENCE NOTE: || removes brms's default intercept/slope correlation;
# the Python priors are separate independent effects, with no LKJ term.
# Gaussian kappa effects use brms's non-centered parameterization and
# positive sd bounds turn normal(0, 2) into HalfNormal(2).
# The nonlinear mu vectors retain centered von Mises effects, not normal
# random effects. Verify their generated names/bounds/priors when baking.
prior_vonmises


## 3. Fit

Full sampling uses four chains with 2,000 warmup and 1,000 retained
draws per chain. CI uses two chains with 1,500 warmup and 1,500 retained
draws. The longer CI adaptation than the introductory GLM and
`adapt_delta = 0.97` allow for the grouped model geometry.


In [ ]:
# Inspect the generated Stan before sampling: each direction coefficient
# must have [-pi, pi] bounds; kappa has independent non-centered effects.
stan_code <- make_stancode(form_vonmises, data = data,
  prior = prior_vonmises,
  stanvars = stan_unwrap_fun + modulo_circular_fun + stan_direction_priors)
stopifnot(grepl("kmu_intercept", stan_code, fixed = TRUE),
          grepl("unwrap_von_mises_vect_lpdf", stan_code, fixed = TRUE))
fit <- brm(form_vonmises, data = data,
  stanvars = stan_unwrap_fun + modulo_circular_fun + stan_direction_priors,
  prior = prior_vonmises, backend = "cmdstanr",
  chains = CHAINS, iter = ITER, warmup = WARMUP, cores = CHAINS,
  seed = SEED, control = list(adapt_delta = ADAPT_DELTA))


## Diagnostics and traces

Report the tutorial's divergence probe first. If flags were not retained,
the output says so; this does not establish zero divergences. Circular
direction coefficients use `Rhat_unwrap`; all diagnostics preserve the
iteration-by-chain structure. Review the summary table and any flagged
parameters, including individual effects and hyperparameters, before
interpreting the posterior. No generating-value column is appropriate
for these real data.


In [ ]:
diagnose_fit <- function(fit, circular_parameters = character()) {
  # NUTS divergences. Not every brms/cmdstanr combination keeps the
  # per-iteration divergence flag in the fit object, so probe for it and
  # report what the fit object actually carries (the R-hat and ESS values
  # below are the primary convergence checks either way).
  divergences <- tryCatch({
    div_flags <- lapply(fit$fit@sim$samples, function(ch) ch[["divergent"]])
    if (any(vapply(div_flags, is.null, logical(1)))) NA_real_
    else sum(unlist(div_flags))
  }, error = function(e) NA_real_)
  if (is.na(divergences)) {
      cat("Divergences: not retained in this fit object (see R-hat/ESS below).\n")
  } else {
      cat("Divergences:", divergences, "\n")
  }
  available <- posterior::variables(posterior::as_draws_array(fit))
  parameters <- available[grepl("^(b_|sd_|r_|kmu_)", available)]
  draws_array <- posterior::subset_draws(posterior::as_draws_array(fit),
                                         variable = parameters)
  out <- do.call(rbind, lapply(parameters, function(p) {
    # Preserve iteration x chain shape: flattening loses between-chain
    # information and would make the R-hat/ESS checks misleading.
    x <- as.matrix(draws_array[, , p])
    angular <- p %in% circular_parameters
    z <- if (angular) mod_circular(x - circ_mean(x)) else x
    stopifnot(all(is.finite(x)))
    data.frame(parameter = p,
      mean = if (angular) circ_mean(x) else mean(x),
      sd = if (angular) circ_sd(x) else sd(as.numeric(x)),
      rhat = if (angular) Rhat_unwrap(x, method = "sine") else posterior::rhat(x),
      ess_bulk = posterior::ess_bulk(z),
      row.names = NULL, check.names = FALSE)
  }))
  out$rhat <- round(out$rhat, 4)
  out$ess_bulk <- round(out$ess_bulk, 1)
  out
}

available <- posterior::variables(posterior::as_draws_array(fit))
circular_parameters <- available[grepl("^b_(beta0|bias0|bias1)_", available)]
summary_frame <- diagnose_fit(fit, circular_parameters)
summary_frame


In [ ]:
summary_frame[!is.finite(summary_frame$rhat) | summary_frame$rhat >= 1.01 |
              !is.finite(summary_frame$ess_bulk) | summary_frame$ess_bulk <= 400, ]


In [ ]:
trace_parameters <- c("b_beta0_Intercept", "b_beta0_PostStimulus",
  "b_kappa_Intercept", "b_kappa_PostStimulus", "kmu_intercept", "kmu_slope")

# as_draws_df is wide: explicitly assemble a long trace table.
draws_df <- posterior::as_draws_df(posterior::subset_draws(
    posterior::as_draws_array(fit), variable = trace_parameters))
trace_df <- do.call(rbind, lapply(trace_parameters, function(p) {
  value <- as.numeric(draws_df[[p]])
  if (p %in% circular_parameters) value <- mod_circular(value - circ_mean(value))
  data.frame(parameter = p, value = value,
             iteration = draws_df[[".iteration"]],
             chain = factor(draws_df[[".chain"]]))
}))
p_trace <- ggplot(trace_df, aes(iteration, value, colour = chain)) +
  geom_line(alpha = 0.6, linewidth = 0.25) +
  facet_wrap(~ parameter, scales = "free_y", ncol = 2) +
  labs(x = "retained iteration", y = "draw (angular traces centered)") +
  theme(legend.position = "bottom")
save_figure(p_trace, "trace", 9, 6)


## 4. Posterior directions and concentration

Evaluate every track in both conditions. The individual mean-direction
panel and the within-individual $R(\kappa)$ panel include 95% HDIs.
Directions are combined with circular arithmetic; concentration is
evaluated on the natural scale after softplus.


In [ ]:
ids <- levels(data$Track)
prediction_data <- expand.grid(Track = ids, PostStimulus = c(0, 1),
                               KEEP.OUT.ATTRS = FALSE, stringsAsFactors = FALSE)
prediction_data$Track <- factor(prediction_data$Track, levels = ids)
prediction_data$condition <- factor(prediction_data$PostStimulus,
                                     levels = c(0, 1), labels = condition_labels)
# Draw x individual-condition matrices. Evaluate each fitted distributional
# predictor explicitly; no arithmetic averaging across the angular seam.
mu <- mod_circular(posterior_linpred(fit, newdata = prediction_data,
                                     transform = FALSE, dpar = "mu"))
kappa <- posterior_linpred(fit, newdata = prediction_data,
                           transform = TRUE, dpar = "kappa")
stopifnot(identical(dim(mu), dim(kappa)), all(kappa > 0))
J <- length(ids)
pre <- seq_len(J)
post <- J + pre
group_alignment <- cbind(
  Mod(rowMeans(exp(1i * mu[, pre, drop = FALSE]))),
  Mod(rowMeans(exp(1i * mu[, post, drop = FALSE]))))
individual_summary <- do.call(rbind, lapply(seq_len(nrow(prediction_data)), function(j) {
  rbind(data.frame(id = as.character(prediction_data$Track[j]),
    condition = prediction_data$condition[j], quantity = "mean direction (rad)",
    as.list(interval_row(mu[, j], angular = TRUE))),
    data.frame(id = as.character(prediction_data$Track[j]),
    condition = prediction_data$condition[j], quantity = "within-individual R(kappa)",
    as.list(interval_row(A1_safe(kappa[, j])))))
}))
p_individual <- ggplot(individual_summary,
  aes(id, mean, colour = condition, group = condition)) +
  geom_pointrange(aes(ymin = low, ymax = high),
    position = position_dodge(width = 0.5), linewidth = 0.25) +
  facet_wrap(~ quantity, scales = "free_y", ncol = 1) +
  scale_colour_manual(values = condition_colors) +
  labs(x = "individual", y = "posterior mean and 95% HDI") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1),
        legend.position = "bottom")
save_figure(p_individual, "individual_posteriors", 10, 6)


In [ ]:
# As in Python's main summary, group direction here is the equal-track
# circular mean of conditional track directions. The PPC below instead
# uses population fixed effects for its group-direction comparison.
group_mu <- cbind(apply(mu[, pre, drop = FALSE], 1, circ_mean),
                   apply(mu[, post, drop = FALSE], 1, circ_mean))
theta <- seq(-pi, pi, length.out = 201)
group_kde <- do.call(rbind, lapply(1:2, function(c) {
  data.frame(theta = theta %% (2 * pi),
    density = vapply(theta, function(t) mean(exp(vm_logpdf(t, group_mu[, c], 500))), numeric(1)),
    condition = condition_labels[c])
}))
p_group_mu <- ggplot(group_kde, aes(theta, density, colour = condition)) +
  geom_line() + coord_polar() +
  scale_x_continuous(limits = c(0, 2 * pi), expand = c(0, 0)) +
  scale_colour_manual(values = condition_colors) +
  labs(x = "group mean direction (rad)", y = "posterior density") +
  theme(legend.position = "bottom")
save_figure(p_group_mu, "group_direction", 5, 5)
alignment_df <- do.call(rbind, lapply(1:2, function(c) {
  data.frame(condition = condition_labels[c],
    as.list(interval_row(group_alignment[, c])))
}))
p_alignment <- ggplot(alignment_df, aes(condition, mean, colour = condition)) +
  geom_pointrange(aes(ymin = low, ymax = high)) +
  scale_colour_manual(values = condition_colors) +
  labs(x = "", y = "between-individual alignment (95% HDI)")
save_figure(p_alignment, "group_alignment", 5, 4)

# Per-track polar direction posteriors, with empirical circular means.
track_kde <- do.call(rbind, lapply(seq_len(nrow(prediction_data)), function(j) {
  data.frame(theta = theta %% (2 * pi),
    density = vapply(theta, function(t) mean(exp(vm_logpdf(t, mu[, j], 500))), numeric(1)),
    Track = prediction_data$Track[j], condition = prediction_data$condition[j])
}))
empirical <- do.call(rbind, lapply(split(data, list(data$Track, data$condition),
  drop = TRUE), function(g) data.frame(Track = g$Track[1],
    condition = g$condition[1], theta = circ_mean(g$HeadingRadians) %% (2 * pi),
    density = 0)))
p_track_mu <- ggplot(track_kde, aes(theta, density, colour = condition)) +
  geom_line() + geom_point(data = empirical, aes(theta, density), shape = 16) +
  coord_polar() + scale_x_continuous(limits = c(0, 2 * pi), expand = c(0, 0)) +
  facet_wrap(~ Track) + scale_colour_manual(values = condition_colors) +
  labs(x = "track mean direction (rad)", y = "posterior density") +
  theme(legend.position = "bottom")
save_figure(p_track_mu, "track_direction_polar", 11, 9)


## Posterior predictive checks

The Python supplemental check compares hierarchical bootstraps of the
observations with posterior response-parameter summaries: group mean
direction, between-individual alignment, and each individual's direction
and within-individual concentration. Resample individuals first, then
trials/windows within each condition. Open markers show data bootstraps;
filled markers show the posterior, with 95% HDIs. The group mean panel
uses the fixed direction effects, matching the Python PPC.


In [ ]:
# Python's supplemental PPC compares data bootstraps with fitted response
# parameters (not with a new stochastic dataset). Reproduce that check,
# then add a conventional replicated-data check below.
B_GROUP <- if (ci_run) 200L else 2000L
B_INDIV <- if (ci_run) 150L else 1500L
ys <- lapply(seq_len(nrow(prediction_data)), function(j) {
  data$HeadingRadians[as.character(data$Track) ==
    as.character(prediction_data$Track[j]) &
    data$PostStimulus == prediction_data$PostStimulus[j]]
})
stopifnot(all(lengths(ys) > 0L))
bootstrap_pair <- function(y, B) {
  z <- replicate(B, mean(exp(1i * y[sample.int(length(y), length(y), replace = TRUE)])))
  list(mu = Arg(z), R = Mod(z))
}
set.seed(SEED + 10L)
boot_individual <- lapply(ys, bootstrap_pair, B = B_INDIV)
boot_group_mu <- boot_group_R <- matrix(0, nrow = B_GROUP, ncol = 2)
for (b in seq_len(B_GROUP)) {
  selected <- sample.int(J, J, replace = TRUE)
  for (c in 1:2) {
    jj <- selected + (c - 1L) * J
    directions <- vapply(jj, function(j) {
      y <- ys[[j]]
      circ_mean(y[sample.int(length(y), length(y), replace = TRUE)])
    }, numeric(1))
    boot_group_mu[b, c] <- circ_mean(directions)
    boot_group_R[b, c] <- resultant(directions)
  }
}
fixed <- as.data.frame(posterior::as_draws_df(fit))
population_mu <- cbind(mod_circular(fixed$b_beta0_Intercept),
  mod_circular(fixed$b_beta0_Intercept + fixed$b_beta0_PostStimulus))
ppc_group <- do.call(rbind, lapply(1:2, function(c) {
  rbind(
    data.frame(condition = condition_labels[c], source = "data bootstrap",
      quantity = "group mean direction (rad)",
      as.list(interval_row(boot_group_mu[, c], TRUE))),
    data.frame(condition = condition_labels[c], source = "posterior",
      quantity = "group mean direction (rad)",
      as.list(interval_row(population_mu[, c], TRUE))),
    data.frame(condition = condition_labels[c], source = "data bootstrap",
      quantity = "between-individual alignment",
      as.list(interval_row(boot_group_R[, c]))),
    data.frame(condition = condition_labels[c], source = "posterior",
      quantity = "between-individual alignment",
      as.list(interval_row(group_alignment[, c]))))
}))
p_group_ppc <- ggplot(ppc_group,
  aes(condition, mean, colour = condition, shape = source)) +
  geom_pointrange(aes(ymin = low, ymax = high),
    position = position_dodge(width = 0.35)) +
  facet_wrap(~ quantity, scales = "free_y") +
  scale_colour_manual(values = condition_colors) +
  scale_shape_manual(values = c("data bootstrap" = 1, "posterior" = 16)) +
  labs(x = "", y = "mean and 95% HDI") + theme(legend.position = "bottom")
save_figure(p_group_ppc, "ppc_group", 8, 4)

ppc_individual <- do.call(rbind, lapply(seq_along(ys), function(j) {
  rbind(data.frame(id = as.character(prediction_data$Track[j]),
    condition = prediction_data$condition[j], source = "data bootstrap",
    quantity = "mean direction (rad)",
    as.list(interval_row(boot_individual[[j]]$mu, TRUE))),
    data.frame(id = as.character(prediction_data$Track[j]),
    condition = prediction_data$condition[j], source = "data bootstrap",
    quantity = "within-individual R(kappa)",
    as.list(interval_row(boot_individual[[j]]$R))),
    transform(individual_summary[individual_summary$id ==
      as.character(prediction_data$Track[j]) &
      individual_summary$condition == prediction_data$condition[j], ],
      source = "posterior"))
}))
p_ind_ppc <- ggplot(ppc_individual,
  aes(id, mean, colour = source, shape = source)) +
  geom_pointrange(aes(ymin = low, ymax = high),
    position = position_dodge(width = 0.4), linewidth = 0.2) +
  facet_grid(quantity ~ condition, scales = "free_y") +
  scale_shape_manual(values = c("data bootstrap" = 1, "posterior" = 16)) +
  labs(x = "individual", y = "mean and 95% HDI") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1),
        legend.position = "bottom")
save_figure(p_ind_ppc, "ppc_individual", 11, 7)


A second check draws replicated angles from each individual's fitted
likelihood at the original sample sizes. It compares resultant lengths
of observed and replicated samples. This captures finite-sample scatter
as well as uncertainty in the response parameters.


In [ ]:
# Same observed sample sizes, each animal's own conditional likelihood.
# Limit simulation draws only; MCMC fitting still uses the budgets above.
set.seed(SEED + 20L)
draw_ids <- sample.int(nrow(mu), min(nrow(mu), if (ci_run) 100L else 400L))
rep_stats <- do.call(rbind, lapply(seq_along(ys), function(j) {
  rep_mu <- matrix(mu[draw_ids, j], nrow = length(draw_ids), ncol = length(ys[[j]]))
  rep_kappa <- matrix(kappa[draw_ids, j], nrow = length(draw_ids), ncol = length(ys[[j]]))
  yrep <- simulate_angles(rep_mu, rep_kappa)
  data.frame(id = as.character(prediction_data$Track[j]),
    condition = prediction_data$condition[j],
    as.list(interval_row(apply(yrep, 1, resultant))),
    observed = resultant(ys[[j]]))
}))
p_replicates <- ggplot(rep_stats, aes(id, mean, colour = condition)) +
  geom_pointrange(aes(ymin = low, ymax = high)) +
  geom_point(aes(y = observed), shape = 1) +
  facet_wrap(~ condition) + scale_colour_manual(values = condition_colors) +
  labs(x = "individual", y = "replicated R, 95% HDI; open = observed") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1),
        legend.position = "bottom")
save_figure(p_replicates, "ppc_replicated", 10, 4)


## Next steps

Inspect diagnostics and both levels of predictive checks before drawing
biological conclusions. Increase sampling if meaningful parameters have
R-hat at or above 1.01 or bulk ESS at or below 400. `UNWRAP_R_CI=1`
checks the complete data/model shape with shorter simulation budgets;
convergence and warning-free execution still require a local bake.

The [model guide](../models.md) and [circular GLMM guide](../math/glmms.md)
explain the distinction between within-individual precision and
between-individual alignment. Re-run all cells to record versions and
regenerate the inline and archival figures.
